# Paso 0: El "Ritual" de McKinney (Carga e Indexación)

Antes de cualquier análisis, debemos decirle a Pandas que el tiempo es el jefe. Sin un DatetimeIndex, las funciones que verás abajo no funcionarán.

In [17]:
import pandas as pd

# 1. Cargar el dataset
df_monitor = pd.read_csv('files_folder/monitoreo_operaciones_2024.csv')

# 2. Convertir y pasar al índice
df_monitor = (
    df_monitor
    .assign(timestamp = lambda x: pd.to_datetime(x['timestamp']))
    .set_index('timestamp')
    .sort_index()
)

print("Indice temporal configurado. La tabla ahora es una serie de tiempo oficial")
print(df_monitor.head())

Indice temporal configurado. La tabla ahora es una serie de tiempo oficial
                    dispositivo servidor_id  latencia_ms  monto_transaccion
timestamp                                                                  
2024-01-01 00:15:17      Mobile      SRV-02       130.39              22.81
2024-01-01 01:08:58      Mobile      SRV-04       138.87               9.21
2024-01-01 01:47:44      Mobile      SRV-03       165.99             108.77
2024-01-01 05:27:55      Mobile      SRV-02       193.37              55.05
2024-01-01 06:00:50      Mobile      SRV-01       110.26              13.40


# Ejemplo 1: Resampling (Visión de Negocio)

Problema Real: El gerente de ventas no quiere ver 10,000 transacciones. Quiere saber cuánto dinero entró por semana para ver si estamos cumpliendo la meta.

Qué se hizo: Usamos .resample('W') (W de Weekly/Semanal) y sumamos.

In [18]:
# 'W' agrupa por semanas. sum() suma los montos de cada semana
reporte_semanal = (
    df_monitor['monto_transaccion']
    .resample('W')
    .sum()
    .to_frame() # Convertimos a DataFrame para que se vea como tabla
)

print(" Ventas totales semanales (2024)")
print(reporte_semanal.head())

 Ventas totales semanales (2024)
            monto_transaccion
timestamp                    
2024-01-07           17630.58
2024-01-14           19431.30
2024-01-21           20688.69
2024-01-28           19199.53
2024-02-04           17754.46


¿Qué pasó? Pandas tomó las 10,000 filas y las comprimió en aproximadamente 52 filas (una por semana). Esto permite ver la Tendencia a largo plazo sin el ruido de los segundos

# Ejemplo 2: Detección de Colapsos (Latencia por Hora)

Problema Real: El equipo de servidores sospecha que el sistema se ralentiza en la noche. Necesitan el promedio de latencia por hora para confirmar el fallo.

Qué se hizo: .resample('H') (H de Hour) y calculamos el promedio (mean).

In [19]:
# Agrupamos por hora y calculamos la media de respuesta del servidor
latencia_por_hora = (
    df_monitor['latencia_ms']
    .resample('h')  # En Pandas 2.2+, 'h' minúscula reemplaza a 'H'
    .mean()
    .fillna(0)
)

print(" Latencia promedio por hora (2024)")
# Filtramos un rango de la noche para ver si sube (Como lo inyectamos)
print(latencia_por_hora.loc['2024-01-01 17:00:00':'2024-01-01 23:00:00'])


 Latencia promedio por hora (2024)
timestamp
2024-01-01 17:00:00     90.850000
2024-01-01 18:00:00    305.300000
2024-01-01 19:00:00    241.523333
2024-01-01 20:00:00      0.000000
2024-01-01 21:00:00    267.535000
2024-01-01 22:00:00      0.000000
2024-01-01 23:00:00      0.000000
Freq: h, Name: latencia_ms, dtype: float64


¿Qué pasó? Verás que los números suben después de las 18:00. Identificaste la Estacionalidad horaria: el patrón repetitivo de lentitud nocturna.

# Ejemplo 3: Análisis de Crecimiento (La técnica del Shift)

Problema Real: "¿Cuánto más dinero ganamos en la transacción actual comparado con la transacción inmediatamente anterior?" Qué se hizo: Usamos .shift(1) para traer el valor de la fila de arriba al nivel de la fila actual.

In [20]:
# Usamos .assign para crear columnas de comparación temporal
df_crecimiento = df_monitor.c(
    monto_anterior = lambda x:x['monto_transaccion'].shift(1), # Trae el monto de la fila anterior
    diferencia_usd = lambda x:x['monto_transaccion']-x['monto_anterior']
)

print(" Comparación transacción actual vs anterior")
print(df_crecimiento[['monto_transaccion', 'monto_anterior', 'diferencia_usd']].head())

 Comparación transacción actual vs anterior
                     monto_transaccion  monto_anterior  diferencia_usd
timestamp                                                             
2024-01-01 00:15:17              22.81             NaN             NaN
2024-01-01 01:08:58               9.21           22.81          -13.60
2024-01-01 01:47:44             108.77            9.21           99.56
2024-01-01 05:27:55              55.05          108.77          -53.72
2024-01-01 06:00:50              13.40           55.05          -41.65


¿Qué pasó? Lograste comparar dos momentos en el tiempo que estaban en filas separadas. Esto es la base para calcular el Crecimiento Porcentual que tanto piden en finanzas

# Ejemplo 4: "Zoom" a un Incidente (Slicing Parcial)

Problema Real: El 15 de marzo a las 10 AM hubo una caída. Necesitas ver exactamente qué dispositivos estaban conectados en ese momento.

Qué se hizo: Usamos el Slicing de Fecha directo sobre el índice.

In [21]:
# Gracias al DatetimeIndex, podemos filtrar con texto simple
incidente_marzo = df_monitor.loc['2024-03-15 10:00:00':'2024-03-15 10:59:59']

print(f"--- REGISTROS DURANTE EL INCIDENTE (10:00 AM) ---")
print(f"Total eventos encontrados: {len(incidente_marzo)}")
print(incidente_marzo['dispositivo'].value_counts())

--- REGISTROS DURANTE EL INCIDENTE (10:00 AM) ---
Total eventos encontrados: 0
Series([], Name: count, dtype: int64)


¿Qué pasó? Hiciste un "drill-down" (profundización). En lugar de recorrer 10,000 filas, Pandas fue directamente al bloque de memoria de esa hora específica. Es ultra eficiente.